# Pastas and Metran example

This notebook shows how output from Pastas time series models can be analyzed using Metran.

In [ ]:
import hydropandas as hpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pastas as ps

import metran

ps.logger.setLevel("ERROR")

metran.show_versions()

## Read data

Load the observed heads from piezometers at different depths at location B21B0214. The outliers (values outside of $5 \sigma$ (std. dev.)) are removed from the time series.

In [ ]:
oc = hpd.read_dino("./data", subdir=".", suffix="_1.csv")

In [ ]:
oc

In [ ]:
oseries = {}

for o in oc.obs:
    name = o.name
    o = o["stand_m_tov_nap"].rename(o.name)

    # remove outliers outside 5*std
    mean = o.median()
    std = o.std()
    mask_outliers = (o - mean).abs() > 5 * std

    ts = o.copy()
    ts.loc[mask_outliers] = np.nan

    # store time series
    oseries[name] = ts

In [ ]:
# sort the names
sorted_names = list(oseries.keys())
sorted_names.sort()
sorted_names

Plot the heads:

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 4))

for name in sorted_names:
    o = oseries[name]
    ftop = oc.loc[name, "screen_top"]
    fbot = oc.loc[name, "screen_bottom"]
    lbl = f"{name} (filter: NAP{ftop:+.1f} - {fbot:+.1f} m)"
    ax.plot(o.index, o, label=lbl)

ax.set_ylabel("stijghoogte (m NAP)")
ax.legend(loc=(0, 1), ncol=2, frameon=False)
ax.set_ylim(top=-0.5)
ax.grid(True)

Load the precipitation and evaporation data from two nearby weather stations

In [ ]:
p = pd.read_csv(
    "./data/RD_338.csv", index_col=[0], parse_dates=True, usecols=["YYYYMMDD", "RD"]
)
e = pd.read_csv(
    "./data/EV24_260.csv", index_col=[0], parse_dates=True, usecols=["YYYYMMDD", "EV24"]
)

Plot precipitation and evaporation time series

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.plot(p.index, p, label="precipitation")
ax.plot(e.index, e, label="evaporation")
ax.set_ylabel("m/day")
ax.legend(loc="best")

## Build time series models

The time series models attempt to simulate the heads using recharge as stress. The recharge is calculated using $R  = P - f \cdot E$, where $R$ is recharge, $P$ is precipitation, $E$ is evaporation and $f$ is factor that is optimized. The model fit results are printed to the console. The model residuals are stored for analysis with Metran.

In [ ]:
# Normalize the index (reset observation time to midnight (the end of the day)).
p.index = p.index.normalize()
e.index = e.index.normalize()

# set tmin/tmax
tmin = "1988-10-14"
tmax = "2005-11-28"

# store models and residuals
models = []
residuals = []

for name in sorted_names:
    # create model
    ml = ps.Model(oseries[name])
    rm = ps.RechargeModel(prec=p, evap=e)
    ml.add_stressmodel(rm)

    # solve model
    ml.solve(tmin=tmin, tmax=tmax, report=False)

    # print fit statistic
    print(name, f"EVP = {ml.stats.evp():.1f}%")

    # store model
    models.append(ml)

    # get residuals
    r = ml.residuals()
    r.name = name
    residuals.append(r)

## Build Metran model

A Metran model is created using the residuals of the time series models. By analyzing the model residuals we can determine for example, whether there is a common pattern in the residuals, which could indicate a missing influence, or a shortcoming in the model structure. Additionally we might be able to analyze whether there are still outliers left in our time series.

In [ ]:
mt = metran.Metran(residuals)
mt.solve()

Plot the specific and common dynamic components

In [ ]:
axes = mt.plots.state_means()

Plot a simulation, including a confidence interval for B21B0214003.

In [ ]:
ax = mt.plots.simulation(mt.snames[2])